# Boiling Bubble with Conjugate Heat Transfer in Solid

## Objective
This 2D-axi simulation is designed to demenstrate the the bubble growth under saturated boiling, with the activation of conjugate heat transfer in solid


## Summary of Initial and Boundary Conditions for fluid:
- **Boundary Conditions**
  - **Dynamics**: 
    - Outlet condition on the top and right sides.
    - Symmetry on the left side.
    - Wall type BC on the bottom
  - **Thermal**: 
    - Adiabatic (Symmetry) conditions on the left.
    - Conjugate heat transfer with solid substrate at the bottom
    - Fixed temperature $T_{{sat}} $ at the top right side.
    - Fixed temperature profils $ T_{z}=T_{sat} +\Delta T_{ONB}(1.-z/\delta) $ at the right side. $\Delta T_{ONB}$ represents the activation temeprature of wall.
  - **Phase**:
    - Apparent angle feeded from microregion model (see model.txt for details).
    - All other boundaries set to symmetry.
- **Initial Conditions:**
  - **Dynamics**: Initial velocity is zero throughout the domain.
  - **Thermal**: A linear temperature profil is imposed within the liquid, ranging from $ T_{sat} $ at $ z=\delta $ to $  T_{sat} +\Delta T_{ONB} $ at $ z=0$.
  - **Phase**: Initial shape of vapor using $R_0$ and $\theta_0$: $ r^2+(z-R_0cos(\theta_0))^2=R_0^2 $.


## Summary of Initial and Boundary Conditions (thermal) for solid:
- **Boundary Conditions**
  - Adiabatic conditions on the left and right sides.
  - Conjugate heat transfer with fluid domain at the top
  - Fixed heat flux $q_{in}$ at the bottom.
- **Initial Conditions**
  -  $T=T_{sat}+\Delta T_{ONB}$ throughout the domain.

![ICs and BCs for Stefan Problem](src/figs/pic_coupling.png)

## Case Specifications
- **Domain Length**: $ r_{max}  = 2.8\, \text{mm} $, $ z_{max}  = 4.8\, \text{mm} $ and $ z_{s, max}  = 0.8\, \text{mm} $
- **Initial Interface Position**: $R_0 = 120 \, \mu m$ and $ \theta_0 = 22.4 \, ^\circ$ at $ t = 0 \, \text{s} $
- **Material Properties**: Properties for water under atmospheric conditions:

|               | $\rho$ $kg/m^3$| $\mu$ $Pa\cdot s$        | $\lambda$ $W/(m\cdot K)$ | $C_p$ $J/(kg\cdot K)$   |
|---------------|----------------|--------------------------|--------------------------|-------------------------|
| **Liquid**    | 958.37         | $2.8 \times 10^{-4}$     | 0.679                    | $4.21 \times 10^3$      |
| **Vapor**     | 0.597          | $1.227 \times 10^{-5}$   | 0.025                    | $2.077 \times 10^3$     |
| **solid**     | 8500           |       -                  | 234                      | 376                     |
| $\sigma = 0.0589$ N/m, $\mathcal{L} = 2.256 \times 10^6$ J/kg                                                  | 

---

The $\Delta T_{ONB}=6.72K$ is set such that subsequent nucleation occurs almost without waiting time. 

In [ ]:
from trustutils import run
import numpy as np

# Declaration of Author and (optionally) date 
run.introduction("L. WEI and TrioCFD team","31/10/2025")

# Declaration of the TRUST version
run.TRUST_parameters("v1.9.7_beta")

In [ ]:
import math
import os

def thermal_boundary_thickness(delta_T):
    """
    Calculates the thermal boundary layer thickness (delta_th)
    as a function of the temperature difference ΔT.

    Parameters:
        delta_T (float): Temperature difference [K]

    Returns:
        float: Thermal boundary layer thickness [m]
    """

    # Constants
    mu = 2.8e-4          # Dynamic viscosity [Pa·s]
    rho = 958.37         # Density [kg/m³]
    lda = 0.679          # Thermal conductivity [W/m·K]
    cp = 4.21e3          # Specific heat capacity [J/kg·K]
    beta_th = 0.0007504815417629351  # Thermal expansion coefficient [1/K]
    g = 9.81              # Gravitational acceleration [m/s²]

    # Derived properties
    nu = mu / rho         # Kinematic viscosity [m²/s]
    alpha = lda / (rho * cp)  # Thermal diffusivity [m²/s]

    # Thermal boundary layer thickness calculation
    delta_th = 7.14 * math.pow(nu * alpha / (g * beta_th * delta_T), 1.0 / 3.0)
    return delta_th

from math import sqrt, pi, floor, log10
def dt_popinet(dx):
    Cp_v = 2.077e3   # Specific heat capacity of vapor (J/kg·K)
    lambda_v = 0.025  # Thermal conductivity of vapor (W/m·K)
    h_lg = 2.256e6
    
    rho_l = 958.37
    Cp_l=4.21e3
    rho_v = 0.597
    sigma = 5.89e-2
    lambda_l = 0.679  

    rho_m = (rho_v+rho_l)/2.
    n_sig_figs = 2
    dt = sqrt((rho_m/pi/sigma)*(dx)**3)
    scale = -int(floor(log10(abs(dt))) - (n_sig_figs - 1))
    return round(dt, scale)


# read one single TEMP file==========================================================================
import pandas as pd
def T_data_frame(file_path):
    data_list = []
    run.saveFileAccumulator(file_path)
    with open(file_path, 'r') as file:
        for line in file:
            if line.startswith(('Time', '-', '\n')):
                continue
            parts = [part.strip('|').strip() for part in line.split('\t')]
            try:
                time, x, y, twall = float(parts[0]), float(parts[1]), float(parts[2]), float(parts[3])
                data_list.append((time, x, y, twall))
            except ValueError:
                continue

    df_parsed = pd.DataFrame(data_list, columns=['Time', 'X', 'Y', 'Twall'])
    return df_parsed

# heat flux ===========================================================================================
# Function to extract time from header lines
def extract_time(line):
    start = line.find("au temps") + 9
    end = line.find(":", start)
    time_str = line[start:end].strip()
    try:
        return float(time_str)
    except ValueError:
        return None
    
# Function to robustly parse a line of data
def parse_line_robust(line):
    # Extracting numerical values following specific keywords
    x_index = line.find('x=') + 2
    surface_face_index = line.find('surface_face(m2)=') + 17
    flux_par_surface_index = line.find('flux_par_surface(W/m2)=') + 23

    x = float(line[x_index:line.find('y=', x_index)].strip())
    surface_face = float(line[surface_face_index:line.find('flux_par_surface', surface_face_index)].strip())
    flux_par_surface = float(line[flux_par_surface_index:line.find('flux(W)=', flux_par_surface_index)].strip())

    return x, surface_face, flux_par_surface


def P_data_frame(input_filepath):
    num_line_effective = 0
    Times = []
    faces_filepath = "face.txt"
    # Open the input file for reading
    run.saveFileAccumulator(input_filepath)
    run.saveFileAccumulator(faces_filepath)
    with open(input_filepath, 'r') as file:
        # Open the faces file for writing lines that start with '# Face'
        with open(faces_filepath, 'w') as faces_file:
            # Open the rest file for writing all other lines
            # Iterate over each line in the input file
            for line in file:
                # Write lines starting with '# Face' to the faces file
                if line.startswith('# Face'):
                    faces_file.write(line)
                    num_line_effective = num_line_effective + 1
                # Write all other lines to the rest file
                else:
                    time = extract_time(line)
                    Times.append(time)                
    unique_elements = list(set(Times))
    average = int(num_line_effective/np.array(unique_elements).size)
    unique_times = np.sort(np.array(unique_elements, dtype=float))
     
    
    data = [] 
    run.saveFileAccumulator(faces_filepath)
    with open(faces_filepath, 'r') as file:
        for line in file:
            parsed_data = parse_line_robust(line)
            if parsed_data:
                data.append(parsed_data)

    os.remove(faces_filepath)
    df = pd.DataFrame(data, columns=['x', 'surface_face', 'flux_par_surface'])
    # Assign time based on the index of each row and the average number of lines per time
    df['Time'] = unique_times[df.index // int(average)]
    return df

In [ ]:

run.reset()
run.initBuildDirectory()
dx = 40e-6
rmax = 3e-3
zmax = 4.5e-3
zsol = 0.9e-3

Nx = int(rmax/dx)+1
Ny = int(zmax/dx)+1
Ns = int(zsol/dx)+1

Rinjection = dx*4.

nprocx = 2
nprocy = 3

theta = 24.22

dt = dt_popinet(dx)*0.1

delatT = 7

delta_th=thermal_boundary_thickness(delatT)

pws = 1.e4

fname = f'M{int(dx*1.e6)}'
name = 'source'
substitutions_dict = {"rmax" : f'{rmax}',
                      "zmax" : f'{zmax}',
                      "zsol" : f'{zsol}',
                      "Nx" : str(Nx),
                      "Ny" : str(Ny),
                      "Ns" : str(Ns),
                      "nprocx" : str(nprocx),
                      "nprocy" : str(nprocy),
                      "total_procs" : str(nprocy*nprocx),
                      "timestep" : f'{dt}',
                      "dT" : f'{delatT:.3g}',
                      "delta_th" : f'{delta_th:.3g}',
                      "Rinjection" : f'{Rinjection:.4g}',
                      "theta" : f'{theta:.3g}',
                      "pws" : f'{pws:.3g}',
                      }


mesh=run.TRUSTCase(".", "mesh.data").copy("mesh.data",targetDirectory=f"{fname}")
mesh.substitute_template(substitutions_dict)
mesh.run()

tc = run.addCaseFromTemplate("source_loc.data"
                         ,targetDirectory=f"{fname}"
                         ,dic=substitutions_dict
                         ,nbProcs=nprocx*nprocy
                         ,targetData=f"{name}.data"
                         )


run.executeCommand(f'cp {run.ORIGIN_DIRECTORY}/src/model.txt {run.BUILD_DIRECTORY}/{fname}/.')

#if (nprocx*nprocy > 1):
    # print("PARALLEL")
    #tc.partition(overwritePartition=False)
            
run.printCases()

In [ ]:
# Run all cases
run.runCases()

# Results
 


In [ ]:
def get_position_bis(fname, name, biaxi=True):
    import numpy as np
    import os, math
    out = f"{fname}/vap_vol.txt"
    if not os.path.exists(f"{run.BUILD_DIRECTORY}/{out}"):
        os.system(f'grep "^Volume_phase_0" {run.BUILD_DIRECTORY}/{fname}/{name}.err | awk \'{{print $4, $2}}\' > {run.BUILD_DIRECTORY}/{out}')
    run.saveFileAccumulator(out)

    
    data = np.loadtxt(f'{run.BUILD_DIRECTORY}/{out}')
    
    # print(data)
    time = data[:,0]
    vol = data[:, 1] 
    if biaxi:
        posi = (np.array(vol)*3./2./math.pi)**(1./3.)
    else:
        posi = (np.array(vol)*4./math.pi)**(1./2.)
    return time, posi
def slice_with_last_array(data, freq):
    return np.concatenate((data[::freq], data[-1:] if data[-1] not in data[::freq] else []))

## Radius vs. time
The post-processing does not distinguish several bubbles so that jumps appear but are due to the appearance/disapearance of bubbles in the domain.
The 2nd bubble nucleates at $t\approx 22ms$, then the first diseapears at 33ms, a third bubble appears just before 45ms.

In [ ]:
import matplotlib.pyplot as plt

fname = f'M{int(dx*1.e6)}'
name = 'source'

t_loc, r_loc = get_position_bis(fname, name, True)
      
plt.plot( t_loc*1.e3, r_loc*1.e3,  label=f'dx={dx*1.e6:.3g}'+r'$\mu m$')
        
        
# plt.xlim([1,1.4])
# plt.ylim([1,1.2])

plt.xlabel(r"$t$ (ms)")
plt.ylabel(r"$R$ (mm)")
plt.legend(loc="best")
# plt.title(itemC)
plt.show()

## Interface shape and interface velocity at detachement 

In [ ]:
from trustutils import visit
colors_line = [(255, 0, 0, 255), (0, 255, 0, 255), (0, 0, 255, 255)]


fname = f'M{int(dx*1.e6)}'
flata = f'{fname}/source_POST1.lata'
index = 0 

def make_fig(it):
    slata = flata.replace("_POST1","")
    visu = visit.Show(flata,"Mesh","INTERFACES")
    visu.visuOptions(["no_databaseinfo"])  
    visu.visitCommand(f'SetActivePlots(({index}))')
    visu.visitCommand(f'm{index}=MeshAttributes()')
    visu.visitCommand(f'm{index}.opaqueMode=m{index}.Off')
    visu.visitCommand(f'm{index}.meshColor={colors_line[index]}')
    visu.visitCommand(f'm{index}.lineWidth = 2')
    visu.visitCommand(f'SetPlotOptions(m{index})')
    visu.visitCommand(f'm{index}.legendFlag = 0')
    visu.visitCommand(f'SetPlotOptions(m{index})')
    
    
    visu.addField(flata,"Vector","VITESSE_SOM_INTERFACES")
    index_v = 1 + index
    visu.visitCommand(f'SetActivePlots(({index_v}))')
    visu.visitCommand(f'm{index_v}=VectorAttributes()')
    visu.visitCommand(f'm{index_v}.useLegend = 0')
    visu.visitCommand(f'm{index_v}.useStride = 1')
    visu.visitCommand(f'm{index_v}.vectorColor = {colors_line[index]}')
    visu.visitCommand(f'SetPlotOptions(m{index_v})')

    visu.addField(flata,"Pseudocolor","TEMPERATURE_THERMIQUE_ELEM_dom_fluide")
    index_v += 1
    visu.visitCommand(f'SetActivePlots(({index_v}))')
    visu.visitCommand(f'm{index_v}=PseudocolorAttributes()')
    visu.visitCommand(f'm{index_v}.minFlag = 1')
    visu.visitCommand(f'm{index_v}.min = 0.')
    visu.visitCommand(f'm{index_v}.maxFlag = 1')
    visu.visitCommand(f'm{index_v}.max = 7.')
    visu.visitCommand(f'SetPlotOptions(m{index_v})')

    visu.addField(slata,"Pseudocolor","TEMPERATURE_ELEM_dom_solide")
    index_v += 1
    visu.visitCommand(f'SetActivePlots(({index_v}))')
    visu.visitCommand(f'm{index_v}=PseudocolorAttributes()')
    visu.visitCommand(f'm{index_v}.minFlag = 1')
    visu.visitCommand(f'm{index_v}.min = 0.')
    visu.visitCommand(f'm{index_v}.maxFlag = 1')
    visu.visitCommand(f'm{index_v}.max = 7.')
    visu.visitCommand(f'SetPlotOptions(m{index_v})')

    visu.visitCommand("V=View2DAttributes()")
    visu.visitCommand(f"V.windowCoords = (0, {rmax/2.}, {-zsol/2.}, {zmax/2.})")
    visu.visitCommand("V.viewportCoords = (0.15, 0.95, 0.1, 0.95) ")
    visu.visitCommand("SetView2D(V)")
    
    # visu.visitCommand('title = CreateAnnotationObject("Text2D")')
    # visu.visitCommand(f'title.text = "{itemC}"')
    # visu.visitCommand(f'title.position = (0.45, 0.9)')
    # visu.visitCommand(f'title.fontBold = 1')
    visit.setFrame(visu, iteration=it)
    visu.plot() 

make_fig(14)
make_fig(15)
make_fig(16)

make_fig(29)
make_fig(30)
make_fig(31)

## Wall heat flux

In [ ]:
fname = f'M{int(dx*1.e6)}'
file_name = 'source_pb2_Diffusion_chaleur.face'
file_path = f'{run.BUILD_DIRECTORY}/{fname}/{file_name}'
run.saveFileAccumulator(f"{fname}/{file_name}")
df_pwall = P_data_frame(file_path)
print(len(df_pwall)%(Nx-1))
time_pwall = df_pwall['Time'].unique()

### CL receeding

In [ ]:
time_pwall = df_pwall['Time'].unique()
df_parsed = df_pwall
unique_time_steps = time_pwall

tdetached = 16.1e-3

t_cible = np.linspace(0, tdetached/4., 4)

num_mar = 0
fig, ax = plt.subplots()
for index, inst in enumerate(t_cible):
    tsp_ind = np.abs(unique_time_steps - t_cible[index]).argmin()
    time_step = unique_time_steps[tsp_ind]
    data_at_time_step = df_parsed[df_parsed['Time'] == time_step]
    plt.plot(data_at_time_step['x']*1.e3, -data_at_time_step['flux_par_surface']/1.e6
             ,  label= f'{round((time_step)*1.e3, 2):.3g} ms'
            )
    num_mar = num_mar+1

    
plt.xlabel(r'$\mathrm{r\ (mm)}$')
plt.ylabel(r'$\mathrm{\phi _{wall}\ (MW/m^2)}$')
# plt.ylim(8, 32)
plt.xlim(0, 1)
plt.legend( )
# plt.yscale('log')
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
# plt.legend(loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.55))
plt.show()

### CL advancing

In [ ]:
time_pwall = df_pwall['Time'].unique()
df_parsed = df_pwall
unique_time_steps = time_pwall

t_cible = np.linspace(tdetached/4.*3.5, tdetached, 4)

num_mar = 0
fig, ax = plt.subplots()
for index, inst in enumerate(t_cible):
    tsp_ind = np.abs(unique_time_steps - t_cible[index]).argmin()
    time_step = unique_time_steps[tsp_ind]
    data_at_time_step = df_parsed[df_parsed['Time'] == time_step]
    plt.plot(data_at_time_step['x']*1.e3, -data_at_time_step['flux_par_surface']/1.e6
             ,  label= f'{round((time_step)*1.e3, 2):.3g} ms'
            )
    num_mar = num_mar+1

    
plt.xlabel(r'$\mathrm{r\ (mm)}$')
plt.ylabel(r'$\mathrm{\phi _{wall}\ (MW/m^2)}$')
# plt.ylim(8, 32)
plt.xlim(0, 1)
plt.legend( )
# plt.yscale('log')
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
# plt.legend(loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.55))
plt.show()

## Wall temperature

In [ ]:
fname = f'M{int(dx*1.e6)}'
file_name = 'source_pb2_sup_twall.face'
file_path = f'{run.BUILD_DIRECTORY}/{fname}/{file_name}'
run.saveFileAccumulator(f"{fname}/{file_name}")
df_twall = T_data_frame(file_path)
print(len(df_twall)%(Nx-1))
time_twall = df_twall['Time'].unique()

### CL receeding

In [ ]:
# time_twall = df_twall['Time'].unique()
unique_time_steps = time_twall
df_parsed = df_twall

t_cible = np.linspace(0, tdetached/4., 4)

num_mar = 0
fig, ax = plt.subplots()

for index, inst in enumerate(t_cible):
    tsp_ind = np.abs(unique_time_steps - t_cible[index]).argmin()
    time_step = unique_time_steps[tsp_ind]
    data_at_time_step = df_parsed[df_parsed['Time'] == time_step]
    plt.plot(data_at_time_step['X']*1.e3, data_at_time_step['Twall']       
             ,  label= f'{round((time_step)*1.e3, 2):.3g} ms'
         #,linestyle='-', 
            )
    num_mar = num_mar+1




plt.xlabel(r'$r$ (mm)')
plt.ylabel(r'$\Delta T^{w}$  (K)' )



# plt.ylim(8, 32)
plt.xlim(0, 1.25)
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.show()

### CL advancing

In [ ]:
# time_twall = df_twall['Time'].unique()
unique_time_steps = time_twall
df_parsed = df_twall

t_cible = np.linspace(tdetached/4.*3.5, tdetached, 4)

num_mar = 0
fig, ax = plt.subplots()

for index, inst in enumerate(t_cible):
    tsp_ind = np.abs(unique_time_steps - t_cible[index]).argmin()
    time_step = unique_time_steps[tsp_ind]
    data_at_time_step = df_parsed[df_parsed['Time'] == time_step]
    plt.plot(data_at_time_step['X']*1.e3, data_at_time_step['Twall']       
             ,  label= f'{round((time_step)*1.e3, 2):.3g} ms'
         #,linestyle='-', 
            )
    num_mar = num_mar+1




plt.xlabel(r'$r$ (mm)')
plt.ylabel(r'$\Delta T^{w}$  (K)' )



# plt.ylim(8, 32)
plt.xlim(0, 1.25)
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.show()

# Data set

In [ ]:
run.dumpDatasetMD(f"M{int(dx*1.e6)}/{name}.data")